In [ ]:
import os

# Replace with the actual address of your Prefect server
#os.environ["PREFECT_API_URL"] = "http://prefect.local/api"
#os.environ["PREFECT_API_URL"] = "http://10.43.153.8:4200/api"
#os.environ["PREFECT_API_URL"] = "http://10.2.97.207:4200/api"

In [2]:
from dask.distributed import Client
import prefect


client = Client()

/opt/conda/lib/python3.12/site-packages/distributed/client.py:1582: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| toolz   | 1.0.0  | 0.12.0    | 0.12.0  |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))


In [3]:
import dask
import dask.distributed
from prefect_dask.task_runners import DaskTaskRunner

/opt/conda/lib/python3.12/site-packages/tzlocal/unix.py:207: UserWarning: Can not find any timezone configuration, defaulting to UTC.
  warnings.warn("Can not find any timezone configuration, defaulting to UTC.")


In [4]:
client

<Client: 'tcp://10.42.4.92:8786' processes=10 threads=10, memory=27.94 GiB>

In [5]:
client.scheduler

<pooled rpc to 'tcp://dask-scheduler:8786'>

In [6]:
task_runner=DaskTaskRunner(client.scheduler)

In [7]:
# # await run_pipeline(task_runner)

# from prefect import flow
# from prefect import task
# import dask

# @task
# def image_single_spw(x):
#     import time
#     time.sleep(1)
    
    
#     ###############################
#     def image_channel_chunk():
#         import time
#         time.sleep(0.1)
#         return 42
    
#     n_cc = 1000
#     return_vals_list = []
#     for i_cc in range(n_cc):
#         delayed_return_val = dask.delayed(image_channel_chunk)()
#         return_vals_list.append(delayed_return_val)
        
#     dask.compute(return_vals_list)
#     ###############################
        
#     return x**2

# @flow(task_runner=task_runner)
# def image_cube(n_spw):
#     futures = [image_single_spw.submit(i) for i in range(n_spw)]
#     return [f.result() for f in futures]

# image_cube(n_spw=16)

In [ ]:
from prefect import flow, task
from prefect_dask.task_runners import DaskTaskRunner
import dask

@task
def image_channel_chunk():
    import time
    time.sleep(0.1)
    return 42

@task
def image_single_spw(x):
    import time
    time.sleep(1)

    n_cc = 1000
    return_vals_list = []
    for i_cc in range(n_cc):
        delayed_return_val = dask.delayed(image_channel_chunk.fn)()  # use .fn to get raw callable
        return_vals_list.append(delayed_return_val)
        
    dask.compute(*return_vals_list)
        
    return x**2

task_runner=DaskTaskRunner(address=client.scheduler.address)

@flow(task_runner=task_runner)
def image_cube(n_spw):
    futures = [image_single_spw.submit(i) for i in range(n_spw)]
    return [f.result() for f in futures]

image_cube(n_spw=16)

21:18:57.985 | WARNING | prefect.client - Your Prefect server is running an older version of Prefect than your client which may result in unexpected behavior. Please upgrade your Prefect server from version 3.2.0 to version 3.4.8 or higher.

21:18:58.345 | INFO    | Flow run 'icy-duck' - Beginning flow run 'icy-duck' for flow 'image-cube'

21:18:58.350 | INFO    | Flow run 'icy-duck' - View at http://10.43.153.8:4200/runs/flow-run/34d9a459-0429-48b3-94b5-db83307e3055

21:18:58.352 | INFO    | prefect.task_runner.dask - Connecting to an existing Dask cluster at tcp://dask-scheduler:8786

/opt/conda/lib/python3.12/site-packages/distributed/client.py:1582: VersionMismatchWarning: Mismatched versions found

+---------+--------+-----------+---------+
| Package | Client | Scheduler | Workers |
+---------+--------+-----------+---------+
| toolz   | 1.0.0  | 0.12.0    | 0.12.0  |
+---------+--------+-----------+---------+
  warnings.warn(version_module.VersionMismatchWarning(msg[0]["warning"]))
2025-07-14 21:18:58,404 - distributed.protocol.pickle - ERROR - Failed to serialize LLGExpr(dsk={'image_single_spw-37f5b026babea2f91b951915f3510834': <Task 'image_single_spw-37f5b026babea2f91b951915f3510834' image_single_spw(, ...)>}).
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/pickle.py", line 60, in dumps
    result = pickle.dumps(x, **dump_kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
_pickle.PicklingError: Can't pickle <function image_single_spw at 0x7f7904d6b600>: it's not the same object as __main__.image_single_s

21:18:58.408 | ERROR   | Flow run 'icy-duck' - Encountered exception during execution: TypeError('Could not serialize object of type LLGExpr', "LLGExpr(dsk={'image_single_spw-37f5b026babea2f91b951915f3510834': <Task 'image_single_spw-37f5b026babea2f91b951915f3510834' image_single_spw(, ...)>})")
Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/pickle.py", line 60, in dumps
    result = pickle.dumps(x, **dump_kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
_pickle.PicklingError: Can't pickle <function image_single_spw at 0x7f7904d6b600>: it's not the same object as __main__.image_single_spw

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/pickle.py", line 65, in dumps
    pickler.dump(x)
_pickle.PicklingError: Can't pickle <function image_single_spw at 0x7f7904d6b600>: it's not the same object as __main__.image_single_spw

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/serialize.py", line 366, in serialize
    header, frames = dumps(x, context=context) if wants_context else dumps(x)
                     ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/serialize.py", line 78, in pickle_dumps
    frames[0] = pickle.dumps(
                ^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/distributed/protocol/pickle.py", line 77, in dumps
    result = cloudpickle.dumps(x, **dump_kwargs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/cloudpickle/cloudpickle.py", line 1537, in dumps
    cp.dump(obj)
  File "/opt/conda/lib/python3.12/site-packages/cloudpickle/cloudpickle.py", line 1303, in dump
    return super().dump(obj)
           ^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/socket.py", line 275, in __getstate__
    raise TypeError(f"cannot pickle {self.__class__.__name__!r} object")
TypeError: cannot pickle 'socket' object

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/opt/conda/lib/python3.12/site-packages/prefect/flow_engine.py", line 765, in run_context
    yield self
  File "/opt/conda/lib/python3.12/site-packages/prefect/flow_engine.py", line 1373, in run_flow_sync
    engine.call_flow_fn()
  File "/opt/conda/lib/python3.12/site-packages/prefect/flow_engine.py", line 785, in call_flow_fn
    result = call_with_parameters(self.flow.fn, self.parameters)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/prefect/utilities/callables.py", line 210, in call_with_parameters
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_1142/2242270905.py", line 32, in image_cube
    futures = [image_single_spw.submit(i) for i in range(n_spw)]
               ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/prefect/tasks.py", line 1321, in submit
    future = task_runner.submit(self, parameters, wait_for)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/prefect_dask/task_runners.py", line 436, in submit
    future = self.client.submit(
             ^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/prefect_dask/client.py", line 74, in submit
    future = super().submit(
             ^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/distributed/client.py", line 2157, in submit
    futures = self._graph_to_futures(
              ^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.12/site-packages/distributed/client.py", line 3357, in _graph_to_futures
    expr_ser = Serialized(*serialize(to_serialize(expr), on_error="raise"))
                           ^

21:18:58.459 | INFO    | Flow run 'icy-duck' - Finished in state Failed('Flow run encountered an exception: TypeError: (\'Could not serialize object of type LLGExpr\', "LLGExpr(dsk={\'image_single_spw-37f5b026babea2f91b951915f3510834\': <Task \'image_single_spw-37f5b026babea2f91b951915f3510834\' image_single_spw(, ...)>})")')

TypeError: ('Could not serialize object of type LLGExpr', "LLGExpr(dsk={'image_single_spw-37f5b026babea2f91b951915f3510834': <Task 'image_single_spw-37f5b026babea2f91b951915f3510834' image_single_spw(, ...)>})")

In [10]:
str(client.scheduler.address)

'tcp://dask-scheduler:8786'

In [ ]:
client.close()